# CNN & Huấn luyện

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import time
import json
from PIL import Image

from data_pipeline import train_loader, test_loader, NUM_CLASSES, test_transforms

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Device : {str(device)}")
print(f"Total number of Chinese character classes to recognize: {NUM_CLASSES}")
print(f"Number of batches in Training set: {len(train_loader)} | Test set: {len(test_loader)}")

In [ ]:
torch.backends.cudnn.benchmark = True

class HanziCNN(nn.Module):
    def __init__(self, num_classes):
        super(HanziCNN, self).__init__()
        
        # 64x64 -> 32x32
        self.block1 = nn.Sequential(
            nn.Conv2d(in_channels=1, out_channels=32, kernel_size=5, padding=2),
            nn.BatchNorm2d(num_features=32), nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2)
        )

        # 32x32 -> 16x16
        self.block2 = nn.Sequential(
            nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, padding=1),
            nn.BatchNorm2d(num_features=64), nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2)
        )

        # 16x16 -> 8x8
        self.block3 = nn.Sequential(
            nn.Conv2d(in_channels=64, out_channels=128, kernel_size=3, padding=1),
            nn.BatchNorm2d(num_features=128), nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2)
        )

        # 8x8 -> 4x4
        self.block4 = nn.Sequential(
            nn.Conv2d(in_channels=128, out_channels=256, kernel_size=3, padding=1),
            nn.BatchNorm2d(num_features=256), nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2)
        )

        # Lớp GAP biến ma trận 4x4 -> vector 256 chiều
        self.gap = nn.AdaptiveAvgPool2d((1, 1))

        self.classifier = nn.Sequential(
            nn.Dropout(p=0.4),
            nn.Linear(in_features=256, out_features=num_classes)
        )

    def forward(self, x):
        x = self.block1(x)
        x = self.block2(x)
        x = self.block3(x)
        x = self.block4(x)
        x = self.gap(x)
        x = torch.flatten(x, 1)
        x = self.classifier(x)
        return x

model = HanziCNN(num_classes=NUM_CLASSES).to(device)

In [ ]:
import torch.optim.lr_scheduler as lr_scheduler 

EPOCHS = 50 
learning_rate = 0.001

# Early stopping after 7 epochs no improvement in validation loss
PATIENCE = 7 

# Reduce LR by half if no improvement in validation loss for 2 epochs
LR_PATIENCE = 2 

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

scheduler = lr_scheduler.ReduceLROnPlateau(
    optimizer, 
    mode='min',           
    factor=0.5,           
    patience=LR_PATIENCE
)

In [ ]:
print(f"BẮT ĐẦU HUẤN LUYỆN (Tối đa {EPOCHS} Epochs - Tích hợp ReduceLROnPlateau & Early Stopping)...")

best_loss = float('inf') 
epochs_no_improve = 0 
best_acc = 0.0 

for epoch in range(EPOCHS):
    start_time = time.time()

    model.train()
    running_loss = 0.0
    correct_train = total_train = 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

        _, predicted = torch.max(outputs.data, 1)
        total_train += labels.size(0)
        correct_train += (predicted == labels).sum().item()

    train_acc = 100 * correct_train / total_train
    train_loss = running_loss / len(train_loader)

    model.eval()
    test_loss = 0.0 
    correct_test = total_test = 0

    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            
            loss = criterion(outputs, labels)
            test_loss += loss.item()

            _, predicted = torch.max(outputs.data, 1)
            total_test += labels.size(0)
            correct_test += (predicted == labels).sum().item()

    test_acc = 100 * correct_test / total_test
    val_loss = test_loss / len(test_loader) 

    scheduler.step(val_loss)

    if val_loss < best_loss: 
        best_loss = val_loss
        best_acc = test_acc 
        torch.save(model.state_dict(), 'hanzi_best_weight.pth')
        epochs_no_improve = 0 
        saved_msg = "⭐ Đã lưu Model (Kỷ lục Loss mới)!"
    else:
        epochs_no_improve += 1
        saved_msg = f"❌ Loss không giảm ({epochs_no_improve}/{PATIENCE})"

    epoch_time = time.time() - start_time
    current_lr = optimizer.param_groups[0]['lr'] 
    
    print(f"Epoch [{epoch+1}/{EPOCHS}] | Time: {epoch_time:.0f}s | LR: {current_lr}")
    print(f"   📉 Train Loss: {train_loss:.4f} | 🎯 Train Acc: {train_acc:.2f}%")
    print(f"   📊 Val Loss:   {val_loss:.4f} | 🏆 Val Acc:   {test_acc:.2f}% -> {saved_msg}")
    print("-" * 50)

    # 3. Kích hoạt Early Stopping
    if epochs_no_improve >= PATIENCE:
        print(f"🛑 Early Stopping activated after {PATIENCE} epochs.")
        print(f"Automatically stopped at Epoch {epoch+1} to lock in the best state.")
        break

print(f"🎉 Final Best Accuracy: {best_acc:.2f}%")
print("The model with the best weights has been saved to 'hanzi_best_weight.pth'")